# 62 — TF-Enhancer-Target Gene Triplet Ranking

Integrates three evidence layers for human-specificity of a regulatory triplet:
1. **DA** (Differential Accessibility): enhancer region, human vs rest (ATAC DESeq2)
2. **DE TF**: transcription factor expression, human vs rest (RNA DESeq2)
3. **DE target gene**: target gene expression, human vs rest (RNA DESeq2)

**Score**: `triplet_score = signed_pval(DA) × signed_pval(DE_TF) × signed_pval(DE_gene)`
where `signed_pval = sign(log2FC) × −log10(padj)`

All-positive → strongly human-specific triplet (more accessible AND TF more expressed AND target more expressed in human).


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import subprocess
import tempfile, os

SCENICPLUS_DIR = Path("/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/scenicplus")
CONSENSUS_DIR  = SCENICPLUS_DIR / "consensus_eregulons"
ANNO_DIR       = Path("/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3")
ATAC_RES_DIR   = ANNO_DIR / "13_deseq2_R_pseudobulk/rna_differential_results"  # for RNA DE
ATAC_DA_DIR    = ANNO_DIR / "13_deseq2_R_pseudobulk/differential_results"
RNA_DIR        = Path("/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2")
HUMAN_PEAKS_BED = ANNO_DIR / "10_final/all_peaks_Human.bed"

OUT_DIR = SCENICPLUS_DIR / "triplet_ranking"
OUT_DIR.mkdir(exist_ok=True)

CELL_TYPES_FOCUS = ["Enterocytes", "Colonocytes", "Goblet.cells", "ECs"]  # adjust to your make.names() names
CONTRAST         = "Div_Human_vs_AllPrimates"   # primary human-specificity contrast

print("Directories set up.")
print("Human peaks BED exists:", HUMAN_PEAKS_BED.exists())

## 1. Load consensus eRegulons for Human

In [ ]:
# Load the most conservative human consensus (all 21 runs)
# Fall back to 90pct if all-runs file is missing
ereg_files = sorted(CONSENSUS_DIR.glob("eRegulons_consensus_Human_*.tsv"))
print("Available consensus files:")
for f in ereg_files: print(f"  {f.name}: {sum(1 for _ in open(f))-1:,} triplets")

# Pick 'all*runs' as primary; 90pct as secondary
primary = next((f for f in ereg_files if "all" in f.name and "runs" in f.name), None)
if primary is None:
    primary = next((f for f in ereg_files if "90pct" in f.name), ereg_files[0])

ereg_human = pd.read_csv(primary, sep="\t")
print(f"\nUsing: {primary.name}")
print(f"  {len(ereg_human):,} triplets | {ereg_human['TF'].nunique()} TFs | "
      f"{ereg_human['Gene'].nunique()} genes | {ereg_human['Region'].nunique()} regions")
print(ereg_human.head(3))

## 2. Map SCENIC+ regions → peak IDs via bedtools intersect

In [ ]:
def parse_region(r):
    """Convert 'chr1:1000-2000' to ('chr1', 1000, 2000)."""
    m = re.match(r'(chr[^:]+):([0-9]+)-([0-9]+)', str(r))
    if m: return m.group(1), int(m.group(2)), int(m.group(3))
    return None, None, None

# Build a BED3+name DataFrame for all unique SCENIC+ regions
unique_regions = ereg_human["Region"].unique()
reg_rows = []
for r in unique_regions:
    chrom, start, end = parse_region(r)
    if chrom:
        reg_rows.append({"chrom": chrom, "start": start, "end": end, "region_key": r})
reg_bed = pd.DataFrame(reg_rows)
print(f"{len(reg_bed):,} unique SCENIC+ regions to map")

# Write temp BED, intersect with human peaks
with tempfile.TemporaryDirectory() as tmpdir:
    query_bed = os.path.join(tmpdir, "query.bed")
    out_bed   = os.path.join(tmpdir, "intersect.bed")
    reg_bed[["chrom","start","end","region_key"]].to_csv(
        query_bed, sep="\t", header=False, index=False)

    result = subprocess.run(
        ["bedtools", "intersect", "-a", query_bed, "-b", str(HUMAN_PEAKS_BED),
         "-wa", "-wb", "-f", "0.5"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("bedtools error:", result.stderr)
    else:
        lines = [l.split("\t") for l in result.stdout.strip().split("\n") if l]
        # cols: chrom,start,end,region_key | peak_chrom,peak_start,peak_end,peak_id,...
        region2peak = {}
        for row in lines:
            if len(row) >= 8:
                region_key = row[3]
                peak_id    = row[7] if len(row) > 7 else f"{row[4]}:{row[5]}-{row[6]}"
                region2peak[region_key] = peak_id
        print(f"Mapped {len(region2peak):,}/{len(unique_regions):,} regions to peak IDs")

ereg_human["peak_id"] = ereg_human["Region"].map(region2peak)
print(f"  Triplets with peak_id: {ereg_human['peak_id'].notna().sum():,}")

## 3. Load DA results (ATAC DESeq2, Human vs AllPrimates)

In [ ]:
def load_da_contrast(da_dir, contrast, cell_type=None):
    """Load DA results from evolutionary branch CSV files."""
    rows = []
    # Pattern: da_dir / ultra_robust_branches / {ct} / {contrast}.csv
    # Also check direct evolutionary CSVs
    search_patterns = [
        da_dir / "evolutionary" / f"*/{contrast}.csv",
        da_dir / "evolutionary" / f"*/{contrast}_*.csv",
    ]
    for pattern in search_patterns:
        for f in Path(da_dir).glob(str(pattern).replace(str(da_dir)+"/","")):
            ct_name = f.parent.name
            if cell_type and ct_name != cell_type: continue
            df = pd.read_csv(f, index_col=0)
            df["peak_id"]   = df.index
            df["cell_type"] = ct_name
            df["contrast"]  = contrast
            rows.append(df)
    return pd.concat(rows) if rows else None

# Load human vs all primates DA for each cell type
# The files are saved per cell_type in evolutionary/
da_rows = []
evo_dir = ATAC_DA_DIR / "evolutionary"
if evo_dir.exists():
    for ct_dir in evo_dir.iterdir():
        if not ct_dir.is_dir(): continue
        f = ct_dir / f"{CONTRAST}.csv"
        if f.exists():
            df = pd.read_csv(f, index_col=0)
            df["peak_id"]   = df.index
            df["cell_type"] = ct_dir.name
            da_rows.append(df)

if da_rows:
    da_df = pd.concat(da_rows, ignore_index=True)
    print(f"DA results: {len(da_df):,} rows across {da_df['cell_type'].nunique()} cell types")
    print(f"  Columns: {list(da_df.columns)}")
else:
    print(f"No DA CSVs found at {evo_dir}/{CONTRAST}.csv")
    print("Note: Run 10b notebooks first, or check the exact path.")
    da_df = None

## 4. Load RNA DE results (DESeq2, Human vs AllPrimates)

In [ ]:
# RNA DE from 61_rna_deseq2.ipynb output
rna_evo_dir = RNA_DIR / "rna_differential_results"
rna_rows = []
if rna_evo_dir.exists():
    for ct_dir in rna_evo_dir.iterdir():
        if not ct_dir.is_dir(): continue
        f = ct_dir / f"{CONTRAST}_RNA_DE.csv"
        if f.exists():
            df = pd.read_csv(f, index_col=0)
            df["gene"]      = df.index
            df["cell_type"] = ct_dir.name
            rna_rows.append(df)

if rna_rows:
    rna_df = pd.concat(rna_rows, ignore_index=True)
    print(f"RNA DE results: {len(rna_df):,} rows across {rna_df['cell_type'].nunique()} cell types")
else:
    print(f"No RNA DE CSVs found. Run 61_rna_deseq2.ipynb first.")
    print(f"Expected path: {rna_evo_dir}/{{cell_type}}/{CONTRAST}_RNA_DE.csv")
    rna_df = None

## 5. Build triplet table and compute ranking scores

In [ ]:
def signed_pval(lfc, padj, min_padj=1e-100):
    """Signed -log10(padj): positive = up in human, negative = down."""
    return np.sign(lfc) * (-np.log10(np.clip(padj, min_padj, 1)))

def score_triplets(ereg_df, da_df, rna_df, cell_type):
    """Join eRegulon triplets with DA and RNA DE scores for one cell type."""
    # Filter eRegulons to this cell type (eRegulons are species-wide; apply to all CTs)
    t = ereg_df[ereg_df["peak_id"].notna()].copy()

    if da_df is not None:
        da_ct = da_df[da_df["cell_type"] == cell_type][
            ["peak_id","log2FoldChange","padj","baseMean"]
        ].rename(columns={"log2FoldChange":"da_lfc","padj":"da_padj","baseMean":"da_base"})
        t = t.merge(da_ct, on="peak_id", how="inner")
        t["da_score"] = signed_pval(t["da_lfc"], t["da_padj"])

    if rna_df is not None:
        rna_ct = rna_df[rna_df["cell_type"] == cell_type][
            ["gene","log2FoldChange","padj"]
        ].rename(columns={"log2FoldChange":"rna_lfc","padj":"rna_padj"})

        # TF expression
        tf_scores = rna_ct.rename(columns={"gene":"TF","rna_lfc":"tf_lfc","rna_padj":"tf_padj"})
        t = t.merge(tf_scores, on="TF", how="left")
        t["tf_score"] = signed_pval(t["tf_lfc"].fillna(0), t["tf_padj"].fillna(1))

        # Target gene expression
        gene_scores = rna_ct.rename(columns={"gene":"Gene","rna_lfc":"gene_lfc","rna_padj":"gene_padj"})
        t = t.merge(gene_scores, on="Gene", how="left")
        t["gene_score"] = signed_pval(t["gene_lfc"].fillna(0), t["gene_padj"].fillna(1))

        # Combined triplet score (product of three signed p-values)
        t["triplet_score"] = t["da_score"] * t["tf_score"] * t["gene_score"]
        # Also save additive version (easier to interpret)
        t["triplet_score_sum"] = t["da_score"] + t["tf_score"] + t["gene_score"]

    t["cell_type"] = cell_type
    return t.sort_values("triplet_score", ascending=False)

if da_df is not None and rna_df is not None:
    all_triplets = []
    for ct in da_df["cell_type"].unique():
        scored = score_triplets(ereg_human, da_df, rna_df, ct)
        all_triplets.append(scored)
        print(f"  {ct}: {len(scored):,} scored triplets")

    triplet_df = pd.concat(all_triplets, ignore_index=True)
    print(f"\nTotal scored triplets: {len(triplet_df):,}")
    print(f"Columns: {list(triplet_df.columns)}")
else:
    print("Waiting for DA and/or RNA DE results — re-run after 10b and 61 notebooks complete.")
    triplet_df = None

## 6. Save ranked triplets and top results

In [ ]:
if triplet_df is not None:
    # Save full ranked table
    out_file = OUT_DIR / f"triplet_ranking_{CONTRAST}_Human.tsv"
    triplet_df.to_csv(out_file, sep="\t", index=False)
    print(f"Saved: {out_file}")

    # Save per cell-type top-100 for easy inspection
    for ct, grp in triplet_df.groupby("cell_type"):
        top100 = grp.nlargest(100, "triplet_score")
        ct_file = OUT_DIR / f"top100_triplets_{ct}_{CONTRAST}.tsv"
        top100.to_csv(ct_file, sep="\t", index=False)

    # Print top 20 human-specific triplets (positive scores) across all cell types
    top = triplet_df[triplet_df["triplet_score"] > 0].nlargest(20, "triplet_score")
    cols = ["TF","Region","Gene","cell_type","da_score","tf_score","gene_score","triplet_score",
            "n_runs_detected","regulation"]
    print("\nTop 20 human-specific triplets (all three layers concordant):")
    print(top[[c for c in cols if c in top.columns]].to_string(index=False))

## 7. Visualize — scatter plots per cell type

In [ ]:
if triplet_df is not None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    for ct, grp in triplet_df.groupby("cell_type"):
        if "da_score" not in grp.columns: continue
        grp = grp.dropna(subset=["da_score","tf_score","gene_score"])
        if len(grp) < 5: continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Panel A: DA enhancer vs DE target gene, colored by TF DE score
        sc = axes[0].scatter(grp["da_score"], grp["gene_score"],
                             c=grp["tf_score"], cmap="RdBu_r",
                             vmin=-5, vmax=5, s=10, alpha=0.5)
        plt.colorbar(sc, ax=axes[0], label="TF DE score (signed -log10 padj)")
        axes[0].set_xlabel("DA score (enhancer, human vs rest)")
        axes[0].set_ylabel("Target gene DE score (human vs rest)")
        axes[0].axhline(0, color="gray", lw=0.5); axes[0].axvline(0, color="gray", lw=0.5)
        axes[0].set_title(f"{ct} — DA vs Gene DE")

        # Panel B: triplet score distribution
        axes[1].hist(grp["triplet_score"], bins=100, color="steelblue", edgecolor="none")
        axes[1].axvline(0, color="red", lw=1)
        axes[1].set_xlabel("Triplet score (product of three signed p-values)")
        axes[1].set_ylabel("Count")
        axes[1].set_title(f"{ct} — Triplet score distribution")

        # Label top 5 triplets
        top5 = grp.nlargest(5, "triplet_score")
        for _, row in top5.iterrows():
            axes[0].annotate(f"{row['TF']}→{row['Gene']}",
                             (row["da_score"], row["gene_score"]),
                             fontsize=6, ha="left",
                             arrowprops=dict(arrowstyle="-", color="black", lw=0.5),
                             xytext=(row["da_score"]+0.3, row["gene_score"]+0.3))

        plt.suptitle(f"TF-Enhancer-Gene Triplet Ranking: {ct} ({CONTRAST})", fontsize=12)
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"triplet_scatter_{ct}_{CONTRAST}.pdf", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved scatter plot for {ct}")

In [ ]:
# Summary statistics
if triplet_df is not None:
    print("\n=== Summary ===")
    print(f"Contrast: {CONTRAST}")
    print(f"Total triplets scored: {len(triplet_df):,}")
    if "triplet_score" in triplet_df.columns:
        pos = triplet_df[triplet_df["triplet_score"] > 0]
        print(f"Human-specific (all 3 layers positive): {len(pos):,} "
              f"({100*len(pos)/len(triplet_df):.1f}%)")
        print(f"\nTop TFs by mean triplet score:")
        tf_rank = (pos.groupby("TF")["triplet_score"]
                   .agg(["mean","count"]).rename(columns={"mean":"mean_score","count":"n_triplets"})
                   .sort_values("mean_score", ascending=False))
        print(tf_rank.head(20).to_string())
    print(f"\nAll results in: {OUT_DIR}")